# A3 移植 (4/8)：M3 —— AICPU 写槽失效

> **一句话结论**：Triton 全扫描在 A3 上发射开销约 25.7 ms/步且与计算零重叠；改成一个 AICPU task 扫描全部单元后，压到 0.55 ms/步。

## 1. 问题是什么

KV 写入热槽后，所有 indexer 组里指向该旧行号的 dgi 项都必须置 `-1`，否则注意力会读到已变内容。

上游是 Triton 全扫描：例如 tokens=8、groups=21 时启动 **2688 个 program**。A3 上每个 program 发射约 11 µs，合计 **~30 ms/call**；stream-timeline 实测它独占一条流、与计算流**零重叠**，25.7 ms/步直接压在关键路径上。

## 2. AICPU 做法

一个 AICPU task 扫描全部 `tokens × groups` 单元（约 4 ns/cell，2688×4096 ≈ 11M cell，共 656 µs）：

- `group_table` 是 int64 **设备地址表**（每行一个 dgi 裸指针），AICPU 直接解引用，不需要 torch 层先 gather；
- `(token, group)` 单元的并发只发生在相同 `(state, value)` 上；多个 token 写同一 state 的同值 `-1` 是**良性竞争**（同值写）；
- **无 capture 门**：图里录的是 AICPU task，捕获期不回退 Triton。

## 3. 成本对比

| 路径 | 耗时（4k bs16） |
| --- | --- |
| Triton 全扫描 | 25.7 → 30 ms/步 |
| AICPU 单 task | **0.55 ms/步** |

## 4. 比喻

从"开 2688 个小工分头去找旧书签"变成"一个工人按清单批量扫一遍表格"，省去了大量分头出发的调度开销。

## 5. 小结

- M3 = Triton 发射税 → AICPU 单 task 扫描；
- 设备地址表裸指针解引用；同值写竞争良性；
- 25.7 → 0.55 ms/步；
- 下一篇讲 M4：AIV 融合 C8 pack。

**下一篇**：`05-m4-aiv-fused-pack.ipynb`
